In [27]:
!pip install optuna

In [28]:
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

url="https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns=["Pregnancies","Glucose","BloodPressure","SkinThickness","Insulin","BMI","DiabetesPedigreeFunction","Age","Outcome"]

#load dataset
df=pd.read_csv(url,names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [29]:
import numpy as np

# replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals=["Glucose","BloodPressure","SkinThickness","Insulin","BMI"]
df[cols_with_missing_vals]=df[cols_with_missing_vals].replace(0,np.nan)

# Impute missing values with mean of respective columns
df.fillna(df.mean(),inplace=True)

#check if there are any remaining missing values
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [30]:
# split into eatures(x) and terget(y)
x=df.drop("Outcome",axis=1)
y=df["Outcome"]

# Split dat into training and test stes( 7:3)
x_train,x_test,y_train,y_ytest=train_test_split(x,y,test_size=0.3,random_state=42)

#Scale the data for better performance
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

print(f"Training set shape: {x_train.shape}")
print(f"Test set shape: {x_test.shape}")

Training set shape: (537, 8)
Test set shape: (231, 8)


In [31]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

#define objective function
def objective(trial):

  #suggest values for  hyperparameters
  n_estimators=trial.suggest_int("n_estimator",50,200)
  max_depth=trial.suggest_int("max_depth",3,20)

  #create RandomForestClassifier and calculate accuracy
  model=RandomForestClassifier(
      n_estimators=n_estimators,
      max_depth=max_depth,
      random_state=42
      )
  #perform 3-fold cross-validation and calculate accuracy
  score=cross_val_score(model,x_train,y_train,cv=3,scoring="accuracy").mean()

  #return mean accuracy
  return score

In [32]:
study=optuna.create_study(direction="maximize",sampler=optuna.samplers.TPESampler())
study.optimize(objective,n_trials=50) #run 50 trials to find best hyperparameter

[I 2026-10-29 10:15:15,978] A new study created in memory with name: no-name-9b5b07af-4596-4901-8c6b-617842502f93
[I 2026-10-29 10:15:17,460] Trial 0 finished with value: 0.7616387337057727 and parameters: {'n_estimator': 172, 'max_depth': 5}. Best is trial 0 with value: 0.7616387337057727.
[I 2026-10-29 10:15:19,490] Trial 1 finished with value: 0.7746741154562384 and parameters: {'n_estimator': 153, 'max_depth': 18}. Best is trial 1 with value: 0.7746741154562384.
[I 2026-10-29 10:15:22,172] Trial 2 finished with value: 0.7672253258845437 and parameters: {'n_estimator': 82, 'max_depth': 15}. Best is trial 1 with value: 0.7746741154562384.
[I 2026-10-29 10:15:23,442] Trial 3 finished with value: 0.7635009310986964 and parameters: {'n_estimator': 76, 'max_depth': 11}. Best is trial 1 with value: 0.7746741154562384.
[I 2026-10-29 10:15:25,085] Trial 4 finished with value: 0.7616387337057727 and parameters: {'n_estimator': 79, 'max_depth': 9}. Best is trial 1 with value: 0.77467411545623

In [33]:
#print best result
print(f"Best Trial accuracy: {study.best_trial.value}")
print(f"Best Hyperparameters: {study.best_trial.params}")

Best Trial accuracy: 0.7821229050279329
Best Hyperparameters: {'n_estimator': 124, 'max_depth': 18}


In [38]:
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

# Get the best hyperparameters from the study
best_params = study.best_trial.params.copy()

# Map "n_estimator" (singular) to "n_estimators" (plural) if it exists
if "n_estimator" in best_params:
    best_params["n_estimators"] = best_params.pop("n_estimator")

# Train a RandomForestClassifier using corrected best hyperparameters
best_model = RandomForestClassifier(**best_params, random_state=42)

# fit model to training data
best_model.fit(x_train, y_train)

# make prediction on test set
y_pred = best_model.predict(x_test)

# calculate accuracy on test set (using y_ytest as defined in cell OfYYi5zd3e97)
test_accuracy = accuracy_score(y_ytest, y_pred)

# print test accuracy
print(f"Test accuracy with best hyperparameters: {test_accuracy:.2f}")

Test accuracy with best hyperparameters: 0.74
